# 2장 — 가까움의 수학: 코사인 유사도 📐 (핵심)

웹 챕터의 두 실험을 코드로 재현하고, **정규화의 지름길**까지 확인합니다.

**이 노트북의 목표**
1. 유클리드 거리의 **길이 오염**을 재현: 자세히 쓴 문서가 부당하게 밀려나는 현장
2. `cos_sim` 을 직접 만들어(내적 ÷ 길이 곱) 오염이 사라짐을 확인
3. **정규화하면 코사인 = 그냥 내적**, 그리고 거리 순위 = 코사인 순위가 됨을 확인

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch2')

## 1. 실험 ① · 길이 오염: 자세히 쓴 죄

단어 빈도 벡터 (x = "연차" 계열 단어 수, y = "신청" 계열 단어 수).
문서 B는 A와 **내용이 같고 3배 자세할 뿐**입니다.

In [ ]:
import numpy as np

q = np.array([1, 1])            # 질문 "연차 신청 어떻게 하나요?"
docs = {
    'A (연차 신청 안내, 짧은 판)':        np.array([2, 2]),
    'B (연차 신청 안내, 3배 자세한 판)':  np.array([6, 6]),
    'C (각종 신청 양식 모음, 연차 아님)': np.array([0, 3]),
}

print('📏 유클리드 거리로 잰 순위 (짧을수록 가까움):')
dist_rank = sorted(docs, key=lambda k: np.linalg.norm(q - docs[k]))
for i, k in enumerate(dist_rank, 1):
    print(f'   {i}위  거리 {np.linalg.norm(q - docs[k]):.2f}  {k}')

# 길이 오염의 재현 — 내용이 같은 B가 딴 주제 C보다도 뒤!
assert dist_rank[-1].startswith('B'), 'B(내용 같음, 3배 분량)가 거리에서는 꼴찌여야 — 길이 오염'
assert dist_rank[1].startswith('C'), '딴 주제 C가 B보다 앞서는 게 이 함정의 핵심'
print('\n⚠️ B는 자세히 썼다는 이유만으로 딴 주제 C보다 밀렸다 (길이 오염)')

## 2. 코사인 유사도: 내적 ÷ 길이 곱

$$\cos\theta = \frac{\vec a \cdot \vec b}{|\vec a|\,|\vec b|}$$

분자(내적, 1부 2장)가 **방향 궁합**을 재고, 분모(길이 곱)가 **길이의 영향을 정확히 상쇄**합니다.

In [ ]:
def cos_sim(a, b):
    """코사인 유사도 = 내적 ÷ (길이 × 길이). 눈금: 같은 방향 1, 직교 0, 반대 -1."""
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print('📐 코사인으로 잰 순위 (1에 가까울수록 비슷):')
cos_rank = sorted(docs, key=lambda k: -cos_sim(q, docs[k]))
for i, k in enumerate(cos_rank, 1):
    print(f'   {i}위  cos {cos_sim(q, docs[k]):.2f}  {k}')

assert np.isclose(cos_sim(q, docs['B (연차 신청 안내, 3배 자세한 판)']), 1.0), 'B는 질문과 같은 방향 — cos = 1.0'
assert cos_sim(q, docs['C (각종 신청 양식 모음, 연차 아님)']) < 0.8, 'C(딴 주제)는 확실히 낮아야'

# 코사인의 두 성질을 직접 확인
b = docs['B (연차 신청 안내, 3배 자세한 판)']
assert np.isclose(cos_sim(q, b), cos_sim(q, 10 * b)), '길이를 10배 해도 코사인은 불변 (분모가 상쇄)'
assert np.isclose(cos_sim(np.array([2, 0]), np.array([0, 5])), 0.0), '직교(90°)면 0 — 무관'
print('\n✔ 확인: B가 A와 동점 1.00으로 복귀. 코사인은 방향(주제)만 보고 길이(분량)는 무시')

## 3. 각도 눈금표: 코사인 감각 만들기

In [ ]:
print(' 사잇각 θ   cos θ    뜻')
for deg, meaning in [(0, '같은 방향, 주제 같음'), (45, '꽤 비슷'), (90, '직교, 무관'),
                     (135, '꽤 다름'), (180, '정반대')]:
    c = np.cos(np.radians(deg))
    bar = '█' * int(round((c + 1) * 10))
    print(f'   {deg:4d}°   {c:+.2f}   {bar:20s} {meaning}')

## 4. 정규화: 실무의 지름길

모든 벡터를 **길이 1**(단위 벡터)로 만들어 두면 분모가 1×1이라 **코사인 = 그냥 내적**이 됩니다.
덤으로 거리 순위와 코사인 순위가 완전히 일치하게 돼요: $|\vec a-\vec b|^2 = 2 - 2\cos\theta$.

In [ ]:
def normalize(v):
    """정규화: 성분을 길이로 나눠 길이 1(단위 벡터)로."""
    return v / np.linalg.norm(v)

qn = normalize(q)
for name, d in docs.items():
    dn = normalize(d)
    dot = qn @ dn                       # 정규화 후엔 내적 한 방
    assert np.isclose(dot, cos_sim(q, d)), '정규화된 내적 = 코사인 유사도'
    # |a-b|² = 2 - 2cosθ 관계식 확인 (거리와 코사인이 한 몸)
    assert np.isclose(np.linalg.norm(qn - dn) ** 2, 2 - 2 * dot), '|a-b|² = 2 - 2cosθ'
    print(f'   내적(정규화 후) {dot:+.2f} = cos  |  거리² {np.linalg.norm(qn - dn) ** 2:.2f} = 2 − 2cos  ← {name}')

print('\n✔ 확인: 정규화하면 코사인 = 내적, 그리고 거리 순위 = 코사인 순위 (범인은 거리가 아니라 길이 오염)')

## 5. 관찰 정리 + 🤖 AI 연결

- **길이 오염**: 주제는 방향에, 분량은 길이에 담긴다. 거리로 재면 분량이 벌점이 된다.
- **코사인 유사도 = 내적 ÷ 길이 곱**: 방향만 남긴다. 눈금: 1(같음) / 0(직교=무관) / −1(반대).
- **정규화**(길이 1) 후엔 코사인 = 내적 한 방. 실무 임베딩 모델 출력이 보통 이 상태.
  이때 cosine / dot / L2 는 같은 순위를 내놓는다(벡터 DB 옵션의 정체, 6장).
- 다시 쓰는 수학: **1부 2장 내적**. 그리고 **4부 6장 어텐션 점수(Q·K)도 같은 내적**.
  LLM 안팎에서 "관련성 = 내적"이라는 같은 수학이 돈다.

**다음 챕터**: 3장(임베딩 공간 탐험). 이 자를 들고 진짜(toy) 임베딩 공간에 들어갑니다.
의미에 "방향"이 있다는 것, 그 유명한 **왕 − 남자 + 여자 ≈ 여왕**이 기다려요.